# Corndel AI6 — eLearning Unit 8.1 Companion Notebook
## Data Cleaning for Robust ML Features

In [ ]:
# pandas, numpy, matplotlib, scikit-learn and pydantic are pre-installed in JupyterLite.
# No additional packages are required for this notebook.
print('Environment ready.')

It is 7pm on a Saturday during February half term, the busiest night of the busiest week of the bowling year. Every lane in a 24-lane centre is booked back-to-back. The machines have been running since 10am with no recovery time between sessions, and many have not had a full service since the Christmas rush.

Your task is to build a model that predicts which pinsetters are likely to fail in the next 24 hours, so a technician can be called before a lane goes dark.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('laneguard_sensor_logs.csv')

print(f"Rows: {df.shape[0]}   Columns: {df.shape[1]}")
print(f"Failure rate: {df['failure_within_24h'].mean():.1%}")
print()
print("Missing values per column:")
print(df.isnull().sum().to_string())

The failure rate is 9.1%, roughly three times a normal operating night, because these machines have been running at their limit since 10am. One column has missing values: `pinsetter_drive_temp`.

**Predict:** Do you think the rows with a missing temperature reading are more likely, less likely, or about as likely to have a failure as the rows where temperature was recorded? Write your reasoning in the cell below before running the next cell.

*Write your answer here.*

In [ ]:
# Compare failure rates between rows with and without a temperature reading
null_rows     = df[df['pinsetter_drive_temp'].isnull()]
non_null_rows = df[df['pinsetter_drive_temp'].notna()]

rate_null    = null_rows['failure_within_24h'].mean()
rate_present = non_null_rows['failure_within_24h'].mean()

# How many of the total failures sit in the null rows?
pct_of_failures = null_rows['failure_within_24h'].sum() / df['failure_within_24h'].sum()

print(f"Failure rate, temp missing:  {rate_null:.1%}")
print(f"Failure rate, temp recorded: {rate_present:.1%}")
print(f"Ratio: {rate_null / rate_present:.1f}x")
print()
print(f"Rows with missing temp contain {pct_of_failures:.0%} of all failures in the dataset.")

The rows with a missing temperature reading fail at 4.4 times the rate of rows with a recorded one.

**Predict:** Given that result, what do you think is causing the temperature values to go missing? Write your reasoning in the cell below before continuing.

*Write your answer here.*

Once you have written your prediction, here are the facts that explain the result:

**1.** The temperature sensor trips its self-protection circuit when the motor surface exceeds approximately 92 degrees Celsius. That is the condition that precedes failure. The sensor goes silent precisely when it has the most to say.

**2.** The column mean is approximately 66 degrees, a normal operating temperature. The rows where the sensor tripped were running more than 25 degrees above that.

**3.** Those rows with missing temperature contain approximately a third of all the failures in the dataset, in only 10% of the rows. They are not randomly absent. They are the most failure-heavy rows in the data.

Hold those three facts. Each of the three cleaning strategies handles those rows differently, and one result is going to be counterintuitive.

The three strategies are:

- **Drop** — remove every training row where temperature is missing
- **Mean imputation** — keep those rows but fill each missing temperature with the column mean
- **Mean + indicator** — fill with the mean and add a column called `temp_was_missing` that records which rows the sensor tripped on

The next two cells set up everything needed to run all three. The first defines the classifier and a helper function that handles training and scoring. The second does the train/test split and computes the fill value: that is the number that will replace every missing temperature reading in the two imputation strategies.

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, recall_score

TARGET = 'failure_within_24h'
BASE   = ['pinsetter_drive_temp', 'pinsetter_chassis_vibration',
          'days_since_service', 'cycles_since_service']

def fit_and_score(X_tr, y_tr, X_te, y_te):
    # Up-weight failure cases: 9% failure rate means the model will ignore
    # them without this correction
    w = np.where(y_tr == 1, (y_tr == 0).sum() / (y_tr == 1).sum(), 1.0)
    # n_estimators=300 gives stable results: the ranking of strategies
    # (drop worst, mean+flag best F1) only holds reliably at this setting
    m = GradientBoostingClassifier(n_estimators=300, learning_rate=0.05,
                                   max_depth=4, subsample=0.8, random_state=42)
    m.fit(X_tr, y_tr, sample_weight=w)
    p = m.predict(X_te)
    return m, f1_score(y_te, p, zero_division=0), \
              recall_score(y_te, p, zero_division=0), p

The helper function `fit_and_score` handles one subtlety worth knowing about. The dataset has a 9.1% failure rate, which means roughly 9 failure rows for every 91 normal ones. A classifier that sees that imbalance will often take the path of least resistance: predict "no failure" for every single row and be right 91% of the time. It never actually learns what failure looks like.

The `sample_weight` argument corrects for this by telling the model to treat each failure row as if it appeared roughly ten times in training. The model can no longer ignore failures: each one costs too much in the optimisation for the model to simply predict zero for everything. This is standard practice with imbalanced classification problems and is not specific to this dataset.

`n_estimators=300` is set deliberately: at lower values the qualitative ordering between strategies can shift.

In [ ]:
# Split before cleaning: all three strategies must face the same test rows
# stratify keeps the same failure rate in train and test as in the full dataset
idx_tr, idx_te = train_test_split(
    df.index, test_size=0.2, random_state=42, stratify=df[TARGET])
train, test = df.loc[idx_tr].copy(), df.loc[idx_te].copy()

# Fill value from training data only, never from test rows
fill = train['pinsetter_drive_temp'].mean()
print(f"Fill value: {fill:.1f} degrees C  |  Test failures: {test[TARGET].sum()}")

**Strategy 1: drop the rows.**

The simplest option. Any training row where temperature is missing is removed entirely. The model never sees those rows.

**Predict:** Given fact 3 above (approximately a third of all failures sit in those 10% of rows), what do you expect to happen to recall when those rows are removed from training? Write your reasoning in the cell below before running.

*Write your answer here.*

In [ ]:
# Drop training rows where temperature was not recorded
train_drop = train.dropna(subset=['pinsetter_drive_temp'])

# Test set: fill its few nulls with the training mean regardless of strategy
# so all three models are evaluated on identical rows
test_drop  = test.copy()
test_drop['pinsetter_drive_temp'] = test_drop['pinsetter_drive_temp'].fillna(fill)

The training set now has no null rows. The test set is unchanged in structure: only its few missing temperature values have been filled so predictions can run. Now train and score.

In [ ]:
m_drop, f1_drop, rec_drop, p_drop = fit_and_score(
    train_drop[BASE], train_drop[TARGET],
    test_drop[BASE],  test_drop[TARGET])

total       = int(test_drop[TARGET].sum())
caught_drop = int((p_drop[test_drop[TARGET] == 1] == 1).sum())
print(f"Drop:  F1={f1_drop:.3f}  Recall={rec_drop:.1%}  "
      f"Caught {caught_drop} of {total} failures")

9 failures caught out of 55. That means 46 failures were not predicted. Those are 46 lanes that will break tonight with no technician called.

Fact 3 from above: the null rows contain approximately a third of all the failures in the dataset, yet the drop strategy removed them from training.

**Interpret:** In one sentence, explain what the model lost when those rows were removed.

*Write your answer here.*

Now run strategy 2: instead of dropping the rows where the sensor tripped, fill each missing temperature with the column mean from the training data.

**Predict:** The drop strategy removed the failure-heavy rows from training. This strategy keeps them but replaces their temperature readings with 66 degrees, a normal operating value. Do you expect recall to improve, stay the same, or get worse? Write your reasoning in the cell below before running.

*Write your answer here.*

In [ ]:
# Replace each missing temperature with 66 degrees, the training mean
train_mean = train.copy()
train_mean['pinsetter_drive_temp'] = train_mean['pinsetter_drive_temp'].fillna(fill)
test_mean  = test.copy()
test_mean['pinsetter_drive_temp']  = test_mean['pinsetter_drive_temp'].fillna(fill)

Every missing temperature in both sets is now 66 degrees. The rows are still there, none were dropped. Train and score.

In [ ]:
m_mean, f1_mean, rec_mean, p_mean = fit_and_score(
    train_mean[BASE], train_mean[TARGET],
    test_mean[BASE],  test_mean[TARGET])

caught_mean = int((p_mean[test_mean[TARGET] == 1] == 1).sum())
print(f"Mean:  F1={f1_mean:.3f}  Recall={rec_mean:.1%}  "
      f"Caught {caught_mean} of {total} failures")

Mean imputation caught 17 failures. Drop caught 9.

Check your prediction from above: did you expect mean imputation to outperform drop? Most people expect drop to be safe and mean to introduce error. The result is the opposite, and the reason is fact 3: the rows the drop strategy removed contained approximately a third of all failure cases in the dataset. Mean imputation kept those rows. The temperature signal in them was wrong, 66 degrees instead of whatever extreme value preceded failure, but the other features still carried enough signal for the model to find some of those failures anyway.

The third strategy does the same fill but adds one more column. Before overwriting the temperature, it records which rows had a missing value by setting a new column called `temp_was_missing` to 1 for those rows and 0 for everything else. The model then has four features about the machine's physical state, plus a fifth that marks exactly the rows where the sensor tripped. Those rows have a failure rate of around 30%, more than four times the rate of rows with a recorded temperature. The model can learn from that pattern directly, independently of whatever imputed temperature value sits alongside it.

**Predict:** Given that the indicator column marks the exact rows with the highest failure rate, do you expect recall to improve, stay the same, or get worse compared to mean imputation alone? Write your reasoning in the cell below before running.

*Write your answer here.*

In [ ]:
# Record which rows had a null before overwriting the temperature
train_flag = train.copy()
train_flag['temp_was_missing'] = train_flag['pinsetter_drive_temp'].isnull().astype(int)
train_flag['pinsetter_drive_temp'] = train_flag['pinsetter_drive_temp'].fillna(fill)

test_flag = test.copy()
test_flag['temp_was_missing'] = test_flag['pinsetter_drive_temp'].isnull().astype(int)
test_flag['pinsetter_drive_temp'] = test_flag['pinsetter_drive_temp'].fillna(fill)

`temp_was_missing` is computed before `fillna` runs. If you reversed that order, the column would be all zeros: the null would already be gone before you checked for it.

In [ ]:
FEATURES_FLAG = BASE + ['temp_was_missing']
m_flag, f1_flag, rec_flag, p_flag = fit_and_score(
    train_flag[FEATURES_FLAG], train_flag[TARGET],
    test_flag[FEATURES_FLAG],  test_flag[TARGET])

caught_flag = int((p_flag[test_flag[TARGET] == 1] == 1).sum())
print(f"Mean + indicator:  F1={f1_flag:.3f}  Recall={rec_flag:.1%}  "
      f"Caught {caught_flag} of {total} failures")

Mean and mean + indicator caught the same number of failures: 17 out of 55. The recall is identical. But the F1 is slightly higher for mean + indicator, which means it made fewer false alarms while catching the same failures.

The same recall does not mean the two models learned the same thing. In Workshop 7 you used SHAP to understand what a model learned: SHAP gives a per-prediction score for each feature, showing how much it pushed the output up or down for that specific row. SHAP is the right tool for this comparison, but it requires a compiled package that is not available in this JupyterLite environment. The cell below uses scikit-learn's built-in `feature_importances_` instead. This gives a global view across the whole model rather than per-prediction scores, so it is less precise than SHAP, but it tells the same story here: look at which features each model found useful, and notice what appears in one plot that does not appear in the other.

In [ ]:
import matplotlib.pyplot as plt

# Feature importances: the model's internal view of which features it relied on most.
# This is the GradientBoostingClassifier's own measure based on how much each feature
# reduces impurity across all the trees. No extra packages required.
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))

for ax, (label, model, features) in zip(axes, [
    ('mean imputation',  m_mean, BASE),
    ('mean + indicator', m_flag, FEATURES_FLAG)]):

    idx = model.feature_importances_.argsort()
    ax.barh([features[j] for j in idx], model.feature_importances_[idx],
            color='#E07B39')
    ax.set_title(f'Feature importance: {label}', fontsize=11)
    ax.set_xlabel('Importance')

plt.tight_layout()
plt.show()

The feature importance plots show what each model relied on most heavily during training.

In the mean imputation model, temperature is the most important feature at 0.43, followed by cycles since service at 0.28. That is technically correct: the model used those columns heavily. But roughly 300 of the temperature values were filled in by the cleaning script, not recorded by the sensor. The model does not know which ones. When the temperature signal is partly fictitious, the model partially compensates by leaning more heavily on cycles since service, the next most available signal. The importance measure cannot tell you that is what happened.

In the mean + indicator model, temperature drops slightly to 0.32, and `temp_was_missing` appears at 0.11. Cycles since service moves up to 0.30, now the joint-top feature alongside temperature. That column contains no temperature data at all, only a 0 or a 1 recording whether a reading was absent. The model has learned that the sensor going silent is itself an event worth acting on, and with that signal available, the weight distribution across all features shifts to reflect what the data actually contains.

**Reflect:** If you showed the mean imputation model's feature importance chart to the centre's operations manager, they would see "temperature is the most important factor." That statement is accurate. Write in the cell below: in what sense is it also misleading, and what would a more honest explanation say?

*Hint: consider what happened to the temperature values in the rows where the sensor tripped, before the model ever saw them.*

*Write your answer here.*

Both models were built after you already knew the outcome. In a production pipeline you want to catch the non-random null before imputation runs, not after you have trained three models and compared the results. Pydantic is a Python library for data validation: you define what a valid data record looks like as a typed class, and it checks every incoming row against that definition before anything downstream in the pipeline touches the data. The class below adds a custom validator for the temperature column that flags the kind of null you saw in this dataset.

One thing to know before reading the code: when a numeric column is loaded from a CSV file, pandas stores missing values as `float('nan')`, not as Python `None`. This is true for standard float columns like `pinsetter_drive_temp`. The `model_validator` at the top of the class converts those `nan` values to `None` before the temperature check runs, so the validator works correctly on real dataframe rows.

In [ ]:
from pydantic import BaseModel, field_validator, model_validator, ValidationError
from typing import Optional

Read the class definition carefully. The tests are in Cell 43, a few cells down.

In [ ]:
class LaneReading(BaseModel):
    lane_id:                     int
    pinsetter_drive_temp:        Optional[float] = None  # permitted to be absent
    pinsetter_chassis_vibration: float
    days_since_service:          int
    cycles_since_service:        int

    @model_validator(mode='before')
    @classmethod
    def convert_nan_to_none(cls, values):
        # Pandas passes NaN for missing floats, not None: normalise before field validation runs
        return {k: None if isinstance(v, float) and np.isnan(v) else v
                for k, v in values.items()}

    @field_validator('pinsetter_drive_temp')
    @classmethod  # required by Pydantic v2
    def flag_non_random_null(cls, v):
        if v is None:
            # Stop here: this null is failure-correlated, not random noise
            raise ValueError(
                "Null temperature: sensor protection circuit may have tripped. "
                "Add temp_was_missing indicator column before imputing.")
        return v

The next cell runs two tests: one with a complete row, one with a row where temperature is missing.

Look at `flag_non_random_null` in the class above. **Predict:** What will happen when the second test passes a row with a missing temperature to `LaneReading`? Write your answer in the cell below.

*Write your answer here.*

Run both tests below. The first passes a complete row: it should print the lane number and temperature with no error. The second passes a row where temperature is missing. Read the output from the second test carefully: the error message should name exactly what the problem is and what to do about it.

In [ ]:
# A complete row should pass without any error
healthy = df.dropna().iloc[0].to_dict()
r = LaneReading(**healthy)
print(f"Valid: lane {r.lane_id} at {r.pinsetter_drive_temp:.1f} degrees C")

# A row with a missing temperature should be stopped before imputation runs
null_row = df[df['pinsetter_drive_temp'].isnull()].iloc[0].to_dict()
try:
    LaneReading(**null_row)
except ValidationError as e:
    print(f"\nStopped before imputation:")
    print(f"  {e.errors()[0]['msg']}")

The validator currently stops the pipeline completely when a null temperature arrives. That is useful during development, but in production you may want to let the row through while still flagging it, passing it to the mean + indicator path rather than rejecting it outright.

**Challenge:** Copy the `LaneReading` class from above into the cell below and rename it `LaneReadingV2`. Then make two changes:

1. Remove the `flag_non_random_null` field validator entirely.
2. Add a new field `temp_was_missing: bool = False` to the class.

Then modify `convert_nan_to_none` so that after converting NaN values to None, it also sets `temp_was_missing` to `True` if `pinsetter_drive_temp` is None. That is one extra line inside the existing method.

Every row should pass validation. A row with a missing temperature should come out with `temp_was_missing=True`. A complete row should come out with `temp_was_missing=False`. No code outside the class should be needed.

*Note: the test cell below uses `**null_row` to pass a dictionary as keyword arguments to the class — the `**` unpacks the dict so each key becomes a named argument. This is standard Python and is the same pattern used throughout this notebook when constructing `LaneReading` objects from dataframe rows.*

In [ ]:
# Write your LaneReadingV2 class here


In [ ]:
# Run this cell to test your solution
null_row = df[df['pinsetter_drive_temp'].isnull()].iloc[0].to_dict()
r = LaneReadingV2(**null_row)
print(f"temp_was_missing={r.temp_was_missing}")  # should be True

healthy = df.dropna().iloc[0].to_dict()
r2 = LaneReadingV2(**healthy)
print(f"temp_was_missing={r2.temp_was_missing}")  # should be False

## What the three strategies actually did

Dropping the rows meant the model was trained on a version of the data with the most failure-heavy rows removed. It never saw those cases, so it learned that failures are less common and less connected to temperature than they actually are. The training data was quieter than the real world.

Mean imputation kept those rows in training, which is why recall improved. But every motor that was running above 92 degrees had its temperature reading replaced with 66 degrees before training ran. 66 is the average across all motors on a normal day. So the model saw those failure-heavy rows, but with temperature values that looked completely unremarkable. The temperature column stopped telling the truth about those specific machines. The model had to find them using the other features instead.

Mean + indicator did the same fill, so the temperature values in those rows are still wrong in exactly the same way. The difference is the extra column. `temp_was_missing` is 1 for every row where the sensor tripped and 0 for everything else. The model now has a direct label for the rows that matter most. It does not need to infer them from vibration or cycle counts. It can read the pattern of absence directly. That is what the feature importance chart showed: the model used `temp_was_missing` as one of its five predictors, even though that column contains no physical measurement at all.

## What Pydantic actually gives you

The three models were all built after you had already seen the data and knew what the null pattern meant. In a live pipeline you do not have that luxury. Data arrives row by row, often from sources you do not fully control, and the cleaning decisions have to be made before training, not after.

Pydantic gives you a place to put those decisions where they are visible, testable, and named. A validator that catches a non-random null is not just a check. It is a statement that you know what kind of null this is and what should happen when one arrives. If the data changes, the validator either breaks in an obvious way or keeps passing, and both outcomes are informative. A cleaning decision written as a comment in a script can be deleted or ignored. One written as a validator with a meaningful error message is much harder to lose track of.

## What you have seen in this notebook

You loaded a real sensor dataset, found that missing values were not random, and traced the reason to a specific physical mechanism: a self-protection circuit that trips above 92 degrees. You trained the same classifier three times on the same raw data, changing only how the missing values were handled. The results were different enough to matter: 9 failures caught versus 17, depending on a single cleaning decision made before any model code ran. You compared what each model learned using feature importance charts, and saw that the mean + indicator model correctly identified the sensor going silent as a meaningful event, while the mean imputation model had no way to represent that. Finally you wrote a Pydantic class that makes that distinction explicit at the start of the pipeline, so the decision is documented in code rather than held only in the memory of the person who made it.

---

*© Corndel. Contact the Curriculum team with any queries.*